# Data Visualization — Part 2 (Python)

**Course:** Data Visualization (LSI-M-2, SS22) · Life Science Informatics, Deggendorf Institute of Technology
**Matriculation No.:** 22203735

This notebook recreates a set of figures from artificial genomics data using
`pandas`, `numpy`, and `matplotlib`:

1. Three alternative visualizations of RNA-binding-protein signals together with a genomic annotation track.
2. A single multi-panel figure combining a scatter plot, a bar plot, and the annotation track.
3. Positional 2-mer (k-mer) counts across a set of DNA sequences.

> **Data note.** The CSV/TXT inputs are course-provided files and are **not**
> committed to this repository. Place them in the `data/` folder next to this
> notebook before running (see `data/README.md`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from pathlib import Path

# All data is loaded relative to this notebook, so the project runs on any
# machine once the files are placed in ./data (no hard-coded C:/ or E:/ paths).
DATA_DIR = Path("data")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = False

## 1. Three visualizations of the same data

The annotation track (exons as grey rectangles, transcripts as horizontal lines on the `+`/`-` strand) is identical across all three versions. In the original notebook this drawing logic was copied four times; here it lives in a single helper, `draw_annotation_track`, so a fix or restyle only has to happen once.

In [ ]:
# Load the two datasets for Exercise 1.
df_ann = pd.read_csv(DATA_DIR / "10_project_data_annotation.csv")
df_sig = pd.read_csv(DATA_DIR / "10_project_data_signals.csv")

print("Annotation:", df_ann.shape)
print(df_ann.head())
print("\nSignals:", df_sig.shape)
print(df_sig.head())

In [ ]:
# Pre-compute the exon and transcript tables once.
# .copy() avoids the pandas SettingWithCopyWarning that the original code hit
# by assigning new columns onto a .loc slice.
ANNOT_HEIGHT = 0.2

exons = df_ann.loc[df_ann["type"] == "exon"].copy()
exons["width"] = exons["stop"] - exons["start"]
exons["y"] = np.where(exons["strand"] == "+", 1, 0)

transcripts = df_ann.loc[df_ann["type"] == "transcript"].copy()
transcripts["y"] = np.where(transcripts["strand"] == "+", 1, 0)

SIGNAL_COLS = ["P1", "P2", "P3", "P4"]


def draw_annotation_track(ax):
    """Draw the genomic annotation track (exons + transcripts) on `ax`.

    Fresh patch artists are built on every call so the same track can be
    reused across several figures (a matplotlib Artist cannot be shared
    between Axes). Generalises over any number of transcripts instead of
    hard-coding the first three rows.
    """
    rects = [
        plt.Rectangle(
            (row.start, row.y - ANNOT_HEIGHT / 2),
            width=row.width,
            height=ANNOT_HEIGHT,
            facecolor="gray",
            edgecolor="gray",
        )
        for row in exons.itertuples()
    ]
    ax.add_collection(PatchCollection(rects, match_original=True))

    for t in transcripts.itertuples():
        ax.hlines(t.y, t.start, t.stop, color="gray")
        ax.vlines([t.start, t.stop], t.y - 0.1, t.y + 0.1, color="gray")

    ax.set_xlim(0, 20000)
    ax.set_ylim(-0.2, 1.2)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["-", "+"])
    ax.set_xlabel("Genomic position")
    ax.set_ylabel("Annotation")
    ax.grid(True, which="major", axis="x", linestyle="--")

### 1.1 Version 1 — one line panel per protein

Each protein signal gets its own stacked line panel, with the annotation track at the bottom. The four near-identical panel blocks in the original are now a single loop.

In [ ]:
def plot_version1(signals, proteins=SIGNAL_COLS):
    fig, axes = plt.subplots(len(proteins) + 1, 1, figsize=(15, 10), sharex=True)
    fig.subplots_adjust(hspace=0)
    x = np.arange(len(signals))

    for ax, protein in zip(axes, proteins):
        ax.plot(x, signals[protein], linewidth=2, zorder=2, color="gray")
        ax.set_xlim(0, 20000)
        ax.set_ylim(0, 1.2)
        ax.set_ylabel(protein)
        ax.grid(True, which="major", axis="x", linestyle="--")

    draw_annotation_track(axes[-1])
    fig.savefig(FIG_DIR / "ex1_version1.png", bbox_inches="tight")
    return fig


plot_version1(df_sig)
plt.show()

### 1.2 Version 2 — heatmap (`pcolormesh`)

The four signals are stacked into a single grey-scale heatmap, with the annotation track below.

In [ ]:
fig, (ax_heat, ax_annot) = plt.subplots(2, 1, figsize=(20, 5), sharex=True)
fig.subplots_adjust(hspace=0)

x = np.arange(len(df_sig))
y = np.arange(len(SIGNAL_COLS))
z = df_sig[SIGNAL_COLS].to_numpy().T  # rows = P1..P4

ax_heat.pcolormesh(x, y, z, shading="nearest", cmap="gray_r", vmin=0, vmax=z.max())
ax_heat.set_xlim(0, 20000)
ax_heat.set_ylim(-1, 4)
ax_heat.set_yticks([0, 1, 2, 3])
ax_heat.set_yticklabels(["P4", "P3", "P2", "P1"])

draw_annotation_track(ax_annot)
fig.savefig(FIG_DIR / "ex1_version2.png", bbox_inches="tight")
plt.show()

### 1.3 Version 3 — overlaid coloured lines

All four signals share one panel, each in its own colour, with a legend.

In [ ]:
fig, (ax_lines, ax_annot) = plt.subplots(2, 1, figsize=(20, 5), sharex=True)
fig.subplots_adjust(hspace=0)

colors = ["#66CDAA", "#4682B4", "#FFD700", "#C0C0C0"]
x = np.arange(len(df_sig))
for protein, color in zip(SIGNAL_COLS, colors):
    ax_lines.plot(x, df_sig[protein], label=protein, linewidth=2, zorder=5, color=color)

ax_lines.set_xlim(0, 20000)
ax_lines.set_ylim(0, 1)
ax_lines.set_yticks([0, 0.5, 1])
ax_lines.set_xlabel("Genomic position")
ax_lines.grid(True, which="major", axis="x", linestyle="--")
ax_lines.legend(loc="upper left")

draw_annotation_track(ax_annot)
fig.savefig(FIG_DIR / "ex1_version3.png", bbox_inches="tight")
plt.show()

### 1.4 Discussion — pros and cons of the three approaches

**Version 1 (line panel per protein)**
*Pros:* gives a quick read of each signal's range, minima/maxima, gaps and clusters; changes at a given genomic position are easy to compare across panels because the x-axis is shared; gridlines help line up detail.
*Cons:* over a wide x-range a single line per panel can be hard to read precisely; a single colour is less engaging; with many proteins the stack of panels grows tall, and exact values at a point are hard to read off.

**Version 2 (heatmap / `pcolormesh`)**
*Pros:* compact; works well in a noisy field because it shows signal and background together; colour encodes magnitude so more series fit in less space; isolated extreme values stand out.
*Cons:* trends are harder to follow than with lines, and absolute values are difficult to recover from shading alone.

**Version 3 (overlaid coloured lines)**
*Pros:* same compactness as the heatmap while keeping line-style trend information; colour separates the four series.
*Cons:* relies on colour to distinguish series, so it is less accessible for colour-vision-deficient readers; with denser data the overlaid lines can occlude one another.

A complementary **box plot** would summarise the distribution of each signal (median, spread, outliers) at a glance and scale to large data, at the cost of hiding the positional/exact values — so it pairs well with, rather than replaces, the views above.

## 2. Multi-panel figure: scatter + bar + annotation

The task asks for **one** figure with multiple panels (the bottom panel reuses the annotation track from Exercise 1). The original notebook produced two separate figures; here a `GridSpec` lays out the scatter (top-left), the grouped bar plot (top-right), and the full-width annotation track (bottom).

In [ ]:
df_scat = pd.read_csv(DATA_DIR / "10_project_data_scatter.csv")
df_bar = pd.read_csv(DATA_DIR / "10_project_data_barplot.csv")

print("Scatter:", df_scat.shape)
print(df_scat.head())
print("\nBar plot data:")
print(df_bar)

In [ ]:
fig = plt.figure(figsize=(16, 9))
gs = fig.add_gridspec(2, 2, height_ratios=[2, 1], hspace=0.3, wspace=0.2)
ax_scatter = fig.add_subplot(gs[0, 0])
ax_bar = fig.add_subplot(gs[0, 1])
ax_annot = fig.add_subplot(gs[1, :])

# --- Scatter (top-left) ---
ax_scatter.scatter(
    df_scat["x1"], df_scat["x2"],
    s=80, alpha=0.5, edgecolors="#000000", color="white",
)
ax_scatter.set_xlabel(r"$X_1$")
ax_scatter.set_ylabel(r"$X_2$")
ax_scatter.set_xlim(6, 14)
ax_scatter.set_ylim(-3, 8)

# --- Grouped bar plot (top-right) ---
# NOTE: these three series are the values shown in the target figure. Confirm
# the column layout of df_bar and pull them from it instead of literals, e.g.
#   bars1, bars2, bars3 = (df_bar[c].tolist() for c in df_bar.columns[1:4])
bars1 = [756, 2411, 577, 743]
bars2 = [619, 2189, 821, 781]
bars3 = [689, 782, 689, 719]
groups = ["XY", "XZ", "YX", "YZ"]

bar_width = 0.25
r1 = np.arange(len(bars1))
r2 = r1 + bar_width
r3 = r2 + bar_width

ax_bar.bar(r1, bars1, color="black", width=bar_width, edgecolor="white", label="Condition")
ax_bar.bar(r2, bars2, color="black", width=bar_width, edgecolor="white")
ax_bar.bar(r3, bars3, color="red", width=bar_width, edgecolor="white", label="Control")
ax_bar.set_ylabel("Number of events")
ax_bar.set_xticks(r2)
ax_bar.set_xticklabels(groups)
ax_bar.legend()

# --- Annotation track (bottom, full width) ---
draw_annotation_track(ax_annot)

fig.savefig(FIG_DIR / "ex2_multipanel.png", bbox_inches="tight")
plt.show()

## 3. Positional 2-mer counts in DNA sequences

Each line of `10_project_data_dna_sequences.txt` is one DNA sequence. For every
2-mer (`AA`, `AC`, …, `TT`) we count, **per position**, how often it starts at
that position across all sequences, then plot the 16 positional count profiles.

A small but real fix over the original: sequences are `strip()`-ed so the
trailing newline does not inflate the sequence length or get counted as part of
a k-mer.

In [ ]:
from itertools import product

seq_path = DATA_DIR / "10_project_data_dna_sequences.txt"
with open(seq_path) as fh:
    sequences = [line.strip() for line in fh if line.strip()]

kmers = ["".join(p) for p in product("ACGT", repeat=2)]
seq_len = len(sequences[0])
kmer_counts = {k: np.zeros(seq_len) for k in kmers}

for seq in sequences:
    for j in range(len(seq) - 1):
        kmer = seq[j:j + 2]
        if kmer in kmer_counts:
            kmer_counts[kmer][j] += 1

fig, ax = plt.subplots(figsize=(18, 6))
for kmer in kmers:
    ax.plot(kmer_counts[kmer], label=kmer)

ax.set_xlim(-5, seq_len + 5)
ax.set_xlabel("Position")
ax.set_ylabel("k-mer count")
ax.legend(loc="upper left", ncol=2)
ax.grid(True)
fig.savefig(FIG_DIR / "ex3_kmer_counts.png", bbox_inches="tight")
plt.show()